# Notebook 06 — Regional Analysis
### Persistent Racial Disparities in U.S. Mortgage Approval: Evidence from 42 Million Applications, 2020–2024

**Author:** Rajveer Singh Pall  
**Institution:** Gyan Ganga Institute of Technology and Sciences  

---

Estimates state-level and census-region racial approval gaps and tests for geographic heterogeneity. Computes the Herfindahl-Hirschman Index (HHI) by state and regresses state-level gaps on structural determinants (Table 6B). Finds that disparities are pervasive nationwide rather than locally concentrated.

**Input:** `data/processed/panel_{year}.csv`  
**Output:** `outputs/tables/table_06*.csv`, `outputs/figures/figure_06*.png`  
**Runtime:** ~10 minutes


In [1]:
"""
NOTEBOOK 06: REGIONAL ANALYSIS
===============================
Geographic heterogeneity in racial approval gaps

KEY QUESTIONS:
1. Do racial approval gaps vary across states?
2. Are gaps regionally clustered?
3. Are disparities pervasive or localized?

OUTPUTS:
- Table 6: State-level approval gaps (top 20 by gap)
- Table 6A: Census region gaps
- Table 6B: Geographic heterogeneity tests
- Figure 6A: State-level gaps (bar chart)
- Figure 6B: Regional trends over time
"""

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Statistics
from scipy import stats
from scipy.stats import f_oneway, kruskal

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("RdYlGn_r")

print("="*70)
print("NOTEBOOK 06: REGIONAL ANALYSIS")
print("="*70)
print("✅ Libraries loaded")



NOTEBOOK 06: REGIONAL ANALYSIS
✅ Libraries loaded


In [2]:
# Paths
PROCESSED_DATA_DIR = Path("../data/processed")
TABLES_DIR         = Path("../outputs/tables")
FIGURES_DIR        = Path("../outputs/figures")

TABLES_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

YEARS      = [2020, 2021, 2022, 2023, 2024]
BLACK_CODE = 3
WHITE_CODE = 5

# Census regions using POSTAL CODES (state_code in CSV is 'MI', 'CA', etc.)
CENSUS_REGIONS = {
    'Northeast': ['CT','ME','MA','NH','NJ','NY','PA','RI','VT'],
    'Midwest':   ['IL','IN','IA','KS','MI','MN','MO','NE','ND','OH','SD','WI'],
    'South':     ['AL','AR','DE','DC','FL','GA','KY','LA','MD','MS',
                  'NC','OK','SC','TN','TX','VA','WV'],
    'West':      ['AK','AZ','CA','CO','HI','ID','MT','NV','NM','OR','UT','WA','WY']
}

# Build postal → region lookup
STATE_TO_REGION = {}
for region_name, states in CENSUS_REGIONS.items():
    for s in states:
        STATE_TO_REGION[s] = region_name

# Postal code → full state name
STATE_NAMES = {
    'AL':'Alabama','AK':'Alaska','AZ':'Arizona','AR':'Arkansas','CA':'California',
    'CO':'Colorado','CT':'Connecticut','DE':'Delaware','DC':'DC','FL':'Florida',
    'GA':'Georgia','HI':'Hawaii','ID':'Idaho','IL':'Illinois','IN':'Indiana',
    'IA':'Iowa','KS':'Kansas','KY':'Kentucky','LA':'Louisiana','ME':'Maine',
    'MD':'Maryland','MA':'Massachusetts','MI':'Michigan','MN':'Minnesota',
    'MS':'Mississippi','MO':'Missouri','MT':'Montana','NE':'Nebraska','NV':'Nevada',
    'NH':'New Hampshire','NJ':'New Jersey','NM':'New Mexico','NY':'New York',
    'NC':'North Carolina','ND':'North Dakota','OH':'Ohio','OK':'Oklahoma',
    'OR':'Oregon','PA':'Pennsylvania','RI':'Rhode Island','SC':'South Carolina',
    'SD':'South Dakota','TN':'Tennessee','TX':'Texas','UT':'Utah','VT':'Vermont',
    'VA':'Virginia','WA':'Washington','WV':'West Virginia','WI':'Wisconsin','WY':'Wyoming'
}

MIN_STATE_APPLICATIONS = 10_000
MIN_BLACK_APPLICATIONS =    500

print("Configuration loaded")
print(f"  Regions defined: {list(CENSUS_REGIONS.keys())}")
print(f"  Total states mapped: {len(STATE_TO_REGION)}")

Configuration loaded
  Regions defined: ['Northeast', 'Midwest', 'South', 'West']
  Total states mapped: 51


In [3]:
print("\n" + "="*70)
print("LOADING DATA")
print("="*70)

def stratified_sample(df, size, black_code=3):
    """Sample preserving race proportions."""
    black_mask = df['applicant_race_1'] == black_code
    n_black    = int(size * black_mask.mean())
    n_white    = size - n_black
    return pd.concat([
        df[black_mask].sample(min(n_black, black_mask.sum()), random_state=42),
        df[~black_mask].sample(min(n_white, (~black_mask).sum()), random_state=42)
    ], ignore_index=True)

dfs = []
for year in YEARS:
    path = PROCESSED_DATA_DIR / f"panel_{year}.csv"
    df   = pd.read_csv(path)
    print(f"{year}: {len(df):,} rows loaded")

    for col in ['income', 'loan_amount', 'property_value', 'approved']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.dropna(subset=['approved', 'income', 'loan_amount',
                            'property_value', 'state_code'])
    dfs.append(stratified_sample(df, 1_000_000))
    del df

df_all = pd.concat(dfs, ignore_index=True)
del dfs

# Race and approval
df_all['black']    = (df_all['applicant_race_1'] == BLACK_CODE).astype(int)
df_all['approved'] = pd.to_numeric(df_all['approved'], errors='coerce')

# state_code is already postal string — just strip whitespace
df_all['state_code'] = df_all['state_code'].astype(str).str.strip().str.upper()

# Map postal code → census region (direct string lookup, no type issues)
df_all['census_region'] = df_all['state_code'].map(STATE_TO_REGION)

before = len(df_all)
df_all = df_all[df_all['census_region'].notna()].copy()
after  = len(df_all)

# LTV (CSV already has ltv column but recompute to be safe)
df_all['ltv'] = pd.to_numeric(df_all['ltv'], errors='coerce')
if df_all['ltv'].isna().mean() > 0.5:
    df_all['ltv'] = (df_all['loan_amount'] / df_all['property_value']).clip(0.01, 2.0) * 100

print(f"\nCombined sample:        {after:,} observations")
print(f"Dropped (territories):  {before - after:,}")
print(f"States covered:         {df_all['state_code'].nunique()}")
print(f"\nRegion counts:")
print(df_all['census_region'].value_counts().to_string())
print(f"\nLTV non-null:           {df_all['ltv'].notna().sum():,} rows")
print("\n✅ Data loaded with correct region mapping")


LOADING DATA
2020: 12,050,951 rows loaded
2021: 12,239,263 rows loaded
2022: 7,755,394 rows loaded
2023: 5,570,382 rows loaded
2024: 5,825,960 rows loaded

Combined sample:        4,988,550 observations
Dropped (territories):  11,450
States covered:         51

Region counts:
census_region
South        2015552
West         1148886
Midwest      1108504
Northeast     715608

LTV non-null:           4,985,725 rows

✅ Data loaded with correct region mapping


In [4]:
"""
STATE-LEVEL APPROVAL GAPS
"""
print("\n" + "="*70)
print("STATE-LEVEL ANALYSIS")
print("="*70)

state_results = []
valid_states  = [s for s in df_all['state_code'].unique() if s in STATE_TO_REGION]
print(f"Valid states to process: {len(valid_states)}")

for state in sorted(valid_states):
    df_s = df_all[df_all['state_code'] == state].copy()

    n_total = len(df_s)
    n_black = (df_s['black'] == 1).sum()
    n_white = (df_s['black'] == 0).sum()

    if n_total < MIN_STATE_APPLICATIONS or n_black < MIN_BLACK_APPLICATIONS:
        continue

    approved_w = df_s[df_s['black']==0]['approved'].dropna()
    approved_b = df_s[df_s['black']==1]['approved'].dropna()

    if len(approved_w) < 10 or len(approved_b) < 10:
        continue

    w = approved_w.mean()
    b = approved_b.mean()
    if pd.isna(w) or pd.isna(b):
        continue

    gap = (w - b) * 100
    se  = np.sqrt(w*(1-w)/len(approved_w) + b*(1-b)/len(approved_b)) * 100

    try:
        _, p = stats.ttest_ind(approved_w, approved_b, equal_var=False)
    except Exception:
        p = np.nan

    state_results.append({
        'State':          state,
        'State_Name':     STATE_NAMES.get(state, state),
        'Region':         STATE_TO_REGION[state],
        'N_Total':        n_total,
        'N_Black':        n_black,
        'Black_Pct':      n_black / n_total * 100,
        'White_Approval': w * 100,
        'Black_Approval': b * 100,
        'Gap_pp':         gap,
        'SE':             se,
        'P_Value':        p
    })

state_df = pd.DataFrame(state_results).sort_values('Gap_pp', ascending=False)
print(f"States analyzed:  {len(state_df)}")
print(f"Gap range:        {state_df['Gap_pp'].min():.2f}pp — "
      f"{state_df['Gap_pp'].max():.2f}pp")
print(f"Mean gap:         {state_df['Gap_pp'].mean():.2f}pp")
print(f"\nTop 10 states by gap:")
print(f"{'State':>6} {'Name':<18} {'Region':12} {'Gap(pp)':>9} "
      f"{'White':>8} {'Black':>8} {'N':>8}")
print("─"*75)
for _, r in state_df.head(10).iterrows():
    print(f"{r['State']:>6} {r['State_Name']:<18} {r['Region']:12} "
          f"{r['Gap_pp']:>9.2f} {r['White_Approval']:>7.1f}% "
          f"{r['Black_Approval']:>7.1f}% {r['N_Total']:>8,}")


STATE-LEVEL ANALYSIS
Valid states to process: 51
States analyzed:  40
Gap range:        3.64pp — 23.47pp
Mean gap:         13.25pp

Top 10 states by gap:
 State Name               Region         Gap(pp)    White    Black        N
───────────────────────────────────────────────────────────────────────────
    MS Mississippi        South            23.47    78.3%    54.8%   41,587
    SC South Carolina     South            23.11    82.2%    59.1%  103,737
    LA Louisiana          South            23.05    78.4%    55.4%   56,489
    AR Arkansas           South            22.23    80.3%    58.1%   44,459
    WI Wisconsin          Midwest          18.86    87.1%    68.2%  111,209
    MI Michigan           Midwest          18.26    83.2%    64.9%  169,190
    AL Alabama            South            18.10    80.5%    62.4%   85,757
    IL Illinois           Midwest          17.66    83.9%    66.2%  167,328
    PA Pennsylvania       Northeast        16.43    81.0%    64.5%  195,708
    NC No

In [5]:
print("\n" + "="*70)
print("TABLE 6: TOP 20 STATES BY GAP")
print("="*70)

def stars(p):
    return "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else ""

table6       = state_df.head(20).copy()
table6["Sig"] = table6["P_Value"].apply(stars)

table6.to_csv(TABLES_DIR / "table_06_state_level_gaps.csv", index=False)
print("✅ Table 6 saved")
print(table6[['State','Region','Gap_pp','SE','Sig']].to_string(index=False))


TABLE 6: TOP 20 STATES BY GAP
✅ Table 6 saved
State    Region    Gap_pp       SE Sig
   MS     South 23.466991 0.511998 ***
   SC     South 23.113748 0.381703 ***
   LA     South 23.047471 0.454484 ***
   AR     South 22.227361 0.729897 ***
   WI   Midwest 18.864368 0.751118 ***
   MI   Midwest 18.264460 0.410035 ***
   AL     South 18.101413 0.401563 ***
   IL   Midwest 17.656438 0.360008 ***
   PA Northeast 16.429571 0.381991 ***
   NC     South 16.098432 0.267305 ***
   MA Northeast 15.646936 0.528202 ***
   GA     South 15.392231 0.236319 ***
   NY Northeast 15.092100 0.378366 ***
   MO   Midwest 14.353507 0.553803 ***
   OH   Midwest 14.016738 0.368231 ***
   TN     South 14.008978 0.424129 ***
   DE     South 13.846235 0.793984 ***
   MD     South 13.823381 0.296497 ***
   IN   Midwest 13.302162 0.492454 ***
   NE   Midwest 13.212957 1.401018 ***


In [6]:
print("\n" + "="*70)
print("SECTION 4.5.1 — STATE-LEVEL DETERMINANTS REGRESSION")
print("="*70)

import statsmodels.api as sm

state_reg_rows = []
for _, row in state_df.iterrows():
    state = row['State']
    sd    = df_all[df_all['state_code'] == state]
    state_reg_rows.append({
        'state_code':      state,
        'state_name':      STATE_NAMES.get(state, state),
        'region':          STATE_TO_REGION.get(state, 'Unknown'),
        'gap_pp':          row['Gap_pp'],
        'black_share_pct': sd['black'].mean() * 100,
        'mean_income':     pd.to_numeric(sd['income'], errors='coerce').mean(),
        'mean_ltv':        pd.to_numeric(sd['ltv'],    errors='coerce').mean(),
        'n_lenders':       sd['lei'].nunique() if 'lei' in sd.columns else np.nan,
        'n_apps':          len(sd)
    })

state_reg_df = pd.DataFrame(state_reg_rows).dropna(subset=['gap_pp'])
print(f"States in regression: {len(state_reg_df)}")
print(f"Gap range: {state_reg_df['gap_pp'].min():.2f}pp — "
      f"{state_reg_df['gap_pp'].max():.2f}pp  "
      f"(mean={state_reg_df['gap_pp'].mean():.2f}pp)")

print("\nTop 10 states by gap:")
print(state_reg_df.nlargest(10, 'gap_pp')[
    ['state_name','gap_pp','black_share_pct','n_lenders']].to_string(index=False))

reg_cols = [c for c in ['black_share_pct','mean_income','mean_ltv','n_lenders']
            if c in state_reg_df.columns
            and state_reg_df[c].notna().sum() > 10]

X_sreg   = sm.add_constant(
    state_reg_df[reg_cols].apply(lambda c: (c - c.mean()) / c.std()))
res_sreg = sm.OLS(state_reg_df['gap_pp'], X_sreg).fit()

print(f"\nOLS: State gap ~ standardized characteristics")
print(res_sreg.summary().tables[1])
print(f"Adj R²: {res_sreg.rsquared_adj:.3f}")
print(f"→ Low R² = observables explain little geographic variation")
print(f"→ Consistent with institutional processes as primary driver")

state_reg_df.to_csv(TABLES_DIR / "table_06_state_determinants.csv", index=False)
print("\n✅ State-level determinants saved")


SECTION 4.5.1 — STATE-LEVEL DETERMINANTS REGRESSION
States in regression: 40
Gap range: 3.64pp — 23.47pp  (mean=13.25pp)

Top 10 states by gap:
    state_name    gap_pp  black_share_pct  n_lenders
   Mississippi 23.466991        29.136509        494
South Carolina 23.113748        18.129501        877
     Louisiana 23.047471        26.369736        550
      Arkansas 22.227361        11.113610        548
     Wisconsin 18.864368         3.519499        677
      Michigan 18.264460         8.462084        798
       Alabama 18.101413        19.776811        711
      Illinois 17.656438        11.091389        879
  Pennsylvania 16.429571         8.516259        928
North Carolina 16.098432        17.046540        996

OLS: State gap ~ standardized characteristics
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              13.2521      0.654     20.257      0.0

In [7]:
print("\n" + "="*70)
print("SECTION 4.5.2 — LENDER CONCENTRATION AND GEOGRAPHIC GAPS")
print("="*70)

hhi_by_state = []
for state in state_reg_df['state_code'].unique():
    sd = df_all[df_all['state_code'] == state]
    if 'lei' not in sd.columns or len(sd) == 0:
        continue
    counts  = sd.groupby('lei').size()
    shares  = counts / counts.sum()
    hhi     = (shares ** 2).sum() * 10000
    hhi_by_state.append({'state_code': state, 'hhi': hhi})

hhi_df       = pd.DataFrame(hhi_by_state)
state_reg_df = state_reg_df.merge(hhi_df, on='state_code', how='left')

print(f"HHI computed for {hhi_df['state_code'].nunique()} states")
print(f"  Mean HHI:   {state_reg_df['hhi'].mean():.0f}")
print(f"  Median HHI: {state_reg_df['hhi'].median():.0f}")

corr = state_reg_df['hhi'].corr(state_reg_df['gap_pp'])
print(f"\nCorrelation(HHI, Gap): {corr:.3f}")

hhi_cols = [c for c in ['hhi','black_share_pct','mean_income','mean_ltv']
            if c in state_reg_df.columns
            and state_reg_df[c].notna().sum() > 5]

X_hhi    = sm.add_constant(
    state_reg_df[hhi_cols].apply(lambda c: (c - c.mean()) / c.std()))
res_hhi  = sm.OLS(state_reg_df['gap_pp'], X_hhi).fit()

print(f"\nOLS: State gap ~ HHI + demographics")
print(res_hhi.summary().tables[1])
print(f"Adj R²: {res_hhi.rsquared_adj:.3f}")

state_reg_df.to_csv(TABLES_DIR / "table_07_concentration_effects.csv", index=False)
print("\n✅ Concentration analysis saved")


SECTION 4.5.2 — LENDER CONCENTRATION AND GEOGRAPHIC GAPS
HHI computed for 40 states
  Mean HHI:   213
  Median HHI: 191

Correlation(HHI, Gap): -0.014

OLS: State gap ~ HHI + demographics
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              13.2521      0.650     20.374      0.000      11.932      14.572
hhi                 0.4697      0.684      0.687      0.497      -0.918       1.857
black_share_pct     2.8391      0.742      3.826      0.001       1.333       4.345
mean_income         0.2454      0.707      0.347      0.731      -1.191       1.682
mean_ltv            0.2179      0.794      0.274      0.785      -1.395       1.830
Adj R²: 0.281

✅ Concentration analysis saved


In [8]:
print("\n" + "="*70)
print("TABLE 6A: REGIONAL GAPS")
print("="*70)

regional = []

for region in ["Northeast","Midwest","South","West"]:
    d = df_all[df_all["census_region"]==region]
    if len(d)==0:
        continue

    w = d[d["black"]==0]["approved"].mean()
    b = d[d["black"]==1]["approved"].mean()
    gap = (w-b)*100

    se = np.sqrt(
        w*(1-w)/(d["black"]==0).sum() +
        b*(1-b)/(d["black"]==1).sum()
    )*100

    regional.append({
        "Region": region,
        "Gap_pp": gap,
        "SE": se,
        "CI_L": gap-1.96*se,
        "CI_U": gap+1.96*se,
        "States": d["state_code"].nunique()
    })

table6a = pd.DataFrame(regional)
table6a.to_csv(TABLES_DIR / "table_06A_regional_gaps.csv", index=False)
print("✅ Table 6A saved")




TABLE 6A: REGIONAL GAPS
✅ Table 6A saved


In [9]:
print("\n" + "="*70)
print("TABLE 6B: GEOGRAPHIC HETEROGENEITY TESTS")
print("="*70)

gap_cv = state_df["Gap_pp"].std() / state_df["Gap_pp"].mean()

groups = [
    state_df[state_df["Region"]==r]["Gap_pp"]
    for r in ["Northeast","Midwest","South","West"]
    if len(state_df[state_df["Region"]==r])>0
]

kw_stat, kw_p = kruskal(*groups)

corr_black, p_black = stats.pearsonr(state_df["Black_Pct"], state_df["Gap_pp"])

tests = pd.DataFrame({
    "Test": ["Coeff. of Variation","Kruskal-Wallis","Corr w/ % Black"],
    "Statistic": [f"{gap_cv:.2f}", f"H={kw_stat:.2f}", f"r={corr_black:.2f}"],
    "P_Value": ["—", f"{kw_p:.4f}", f"{p_black:.4f}"]
})

tests.to_csv(TABLES_DIR / "table_06B_geographic_tests.csv", index=False)
print("✅ Table 6B saved")




TABLE 6B: GEOGRAPHIC HETEROGENEITY TESTS
✅ Table 6B saved


In [12]:
# ── Compute regional trends by year ───────────────────────────────────────────
regional_trends_rows = []

for year in YEARS:
    df_yr = df_all[df_all['year'] == year]
    for region in ['Northeast', 'Midwest', 'South', 'West']:
        d = df_yr[df_yr['census_region'] == region]
        if len(d) == 0:
            continue
        w = d[d['black'] == 0]['approved'].mean()
        b = d[d['black'] == 1]['approved'].mean()
        if pd.isna(w) or pd.isna(b):
            continue
        regional_trends_rows.append({
            'Region': region,
            'Year':   year,
            'Gap_pp': (w - b) * 100
        })

regional_trends = pd.DataFrame(regional_trends_rows)
print("✅ regional_trends computed")
print(regional_trends.to_string(index=False))

# ── Plot ───────────────────────────────────────────────────────────────────────
region_line_colors = {
    'Northeast': '#1565C0',
    'Midwest':   '#B71C1C',
    'South':     '#E65100',
    'West':      '#1B5E20',
}

fig, ax = plt.subplots(figsize=(10, 6))

for region in ['Northeast', 'Midwest', 'South', 'West']:
    d = regional_trends[regional_trends['Region'] == region].sort_values('Year')
    if len(d) == 0:
        continue
    color = region_line_colors.get(region, '#555555')
    ax.plot(d['Year'], d['Gap_pp'],
            marker='o', linewidth=2.5, markersize=8,
            color=color, label=region)

ax.set_xticks(YEARS)
ax.set_xticklabels([str(y) for y in YEARS], fontsize=10)
ax.set_xlim(YEARS[0] - 0.3, YEARS[-1] + 0.3)
ax.set_xlabel('Year', fontsize=11, fontweight='bold')
ax.set_ylabel('Racial Approval Gap (pp)', fontsize=11, fontweight='bold')
ax.set_title('Racial Approval Gaps by Census Region (2020–2024)',
             fontsize=12, fontweight='bold', pad=12)
ax.legend(fontsize=10, framealpha=0.95, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "figure_06B_regional_trends.png", dpi=300, bbox_inches='tight')
plt.close()
print("✅ figure_06B_regional_trends.png saved")

✅ regional_trends computed
   Region  Year    Gap_pp
Northeast  2020 13.704154
  Midwest  2020 14.728880
    South  2020 13.695069
     West  2020  7.361129
Northeast  2021 11.620457
  Midwest  2021 13.595399
    South  2021 12.404466
     West  2021  6.918161
Northeast  2022 14.223502
  Midwest  2022 14.538848
    South  2022 13.569449
     West  2022  9.634435
Northeast  2023 15.772470
  Midwest  2023 16.518672
    South  2023 14.239264
     West  2023  9.592086
Northeast  2024 14.879530
  Midwest  2024 16.647052
    South  2024 13.752544
     West  2024  9.489761
✅ figure_06B_regional_trends.png saved


In [13]:
# Region line colors — distinct, colorblind-friendly
region_line_colors = {
    'Northeast': '#1565C0',   # Dark blue
    'Midwest':   '#B71C1C',   # Dark red
    'South':     '#E65100',   # Dark orange
    'West':      '#1B5E20',   # Dark green
}

fig, ax = plt.subplots(figsize=(10, 6))

rt = pd.DataFrame(regional_trends)   # your computed data from above

for region in rt['Region'].unique():
    d = rt[rt['Region'] == region].sort_values('Year')
    color = region_line_colors.get(region, '#555555')
    ax.plot(d['Year'], d['Gap_pp'],
            marker='o', linewidth=2.5, markersize=8,
            color=color, label=region)

# ── FIX: integer x-axis ticks ─────────────────────────────────────────────────
ax.set_xticks(YEARS)
ax.set_xticklabels([str(y) for y in YEARS], fontsize=10)
ax.set_xlim(YEARS[0] - 0.3, YEARS[-1] + 0.3)

ax.set_xlabel('Year', fontsize=11, fontweight='bold')
ax.set_ylabel('Racial Approval Gap (pp)', fontsize=11, fontweight='bold')
ax.set_title('Racial Approval Gaps by Census Region (2020–2024)',
             fontsize=12, fontweight='bold', pad=12)
ax.legend(fontsize=10, framealpha=0.95, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "figure_06B_regional_trends.png", dpi=300, bbox_inches='tight')
plt.close()
print("✅ figure_06B_regional_trends.png saved")

✅ figure_06B_regional_trends.png saved


In [14]:
print("\n" + "="*80)
print("SUMMARY FOR MANUSCRIPT – REGIONAL ANALYSIS")
print("="*80)

print(f"""
Across {len(state_df)} states, racial approval gaps are present nationwide.
The mean state-level gap is {state_df['Gap_pp'].mean():.2f}pp, with moderate
geographic dispersion (CV = {gap_cv:.2f}).

Regional averages differ modestly, but no census region is free of disparities.
Statistical tests indicate limited regional clustering, consistent with
pervasive rather than localized discrimination.
""")




SUMMARY FOR MANUSCRIPT – REGIONAL ANALYSIS

Across 40 states, racial approval gaps are present nationwide.
The mean state-level gap is 13.25pp, with moderate
geographic dispersion (CV = 0.37).

Regional averages differ modestly, but no census region is free of disparities.
Statistical tests indicate limited regional clustering, consistent with
pervasive rather than localized discrimination.

